# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and fields by their `@id` fields.

In [ ]:
# List all record sets in the dataset, referencing their @id
print('Available record sets and their fields:')

from pprint import pprint

record_sets = metadata.record_sets
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    # Each record set may contain several fields
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields (by @id):")
        for field in fields:
            if isinstance(field, dict) and '@id' in field:
                print(f"    {field['@id']}")
            else:
                print(f"    {field}")
    else:
        print("  [No explicit fields found]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Reference record sets and fields by their @id

# Gather all record set @id's
record_set_ids = [rs['@id'] for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Load records for each record set
    print(f"Loading records for Record Set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Loaded {len(df)} records.")
            print(f"  Columns: {df.columns.tolist()}")
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error loading records: {e}")

# For demonstration, pick the first non-empty record set
main_record_set = None
for rid in record_set_ids:
    if rid in dataframes:
        main_record_set = rid
        break

if main_record_set is not None:
    print(f"\nUsing record set '{main_record_set}' for further analysis.")
    display(dataframes[main_record_set].head())
else:
    print("No record set with data available.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping/categorizing data.

In [ ]:
import numpy as np

if main_record_set is None:
    print("No data for EDA.")
else:
    df = dataframes[main_record_set]
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()

    # If there are no numeric fields, try to infer numeric from object
    if not numeric_fields:
        # Try to convert object fields to numeric
        candidate_numeric = []
        for col in df.columns:
            try:
                _ = pd.to_numeric(df[col])
                candidate_numeric.append(col)
            except Exception:
                continue
        numeric_fields = candidate_numeric

    if not numeric_fields:
        print("No numeric fields available for analysis.")
    else:
        # Pick the first numeric field by @id
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field for analysis: {numeric_field_id}")

        # Filter: values > threshold, set threshold as a quartile
        try:
            field_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
            threshold = field_series.quantile(0.75)  # e.g., top 25%
            filtered_df = df.loc[field_series > threshold].copy()
        except Exception as e:
            print(f"Error processing numeric values: {e}")
            filtered_df = pd.DataFrame()

        if not filtered_df.empty:
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            display(filtered_df.head())

            # Normalize
            filtered_df[f"{numeric_field_id}_normalized"] = (
                pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - field_series.mean()
            ) / field_series.std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Try grouping by a categorical field
            group_candidates = df.select_dtypes(include=[object]).columns.tolist()
            group_field_id = None
            for g in group_candidates:
                if g != numeric_field_id and df[g].nunique() > 1 and df[g].nunique() < 20:
                    group_field_id = g
                    break
            if group_field_id:
                print(f"Grouping by: {group_field_id}")
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped data by {group_field_id}:")
                display(grouped_df.head())
        else:
            print("No records passed the filter.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if main_record_set is None:
    print("No data available for visualization.")
elif not numeric_fields:
    print("No numeric fields for visualization.")
else:
    # Histogram of numeric field
    plt.figure(figsize=(7, 4))
    try:
        plt.hist(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=20)
        plt.title(f"Distribution of '{numeric_field_id}'")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()
    except Exception as e:
        print(f"Error plotting histogram: {e}")

    # If grouping succeeded
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(8, 4))
        try:
            plt.bar(grouped_df[group_field_id], grouped_df[numeric_field_id])
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.xticks(rotation=45)
            plt.show()
        except Exception as e:
            print(f"Error plotting group bar chart: {e}")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset using `mlcroissant`, referencing all entities by their `@id` fields. We previewed the available record sets and fields, extracted data for a selected record set, applied basic EDA steps (such as filtering by value, normalization, and grouping), and visualized key distributions. This workflow demonstrates reusable, standards-based access and analysis for Croissant-format datasets.